# D5 — Stability and inter-explainer agreement (H6)

**H6 (registered: SHAP and LIME disagree materially on per-finding attributions)** was operationalised in advance. For each E-rand CVE, compute Kendall's τ-b between |TreeSHAP| and |LIME| over all 69 features (M-XGB, seed 0). Take the median over 1,000 CVEs, with a 95% bootstrap CI (2,000 resamples). **Supported if the CI's upper bound is < 0.5**, not supported if its lower bound is ≥ 0.5, inconclusive otherwise.

Noise ceilings:
- LIME against itself across 10 seeds (first 200 E-rand CVEs);
- TreeSHAP across 10 XGBoost training seeds;
- a Wilcoxon test of SHAP–LIME τ against LIME–LIME τ.

Design: Study D addendum (commit c0a3ee1), registered before any Study D explanation was computed.

Reads saved outputs. Run: `python -m src.explainability.stability` (after D2, ~5 min).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

## H6

In [2]:
pd.read_csv("outputs/d5_h6.csv").T

,0
statistic,"median Kendall tau-b |TreeSHAP| vs |LIME|, M-X..."
median,0.349531
ci_low,0.344416
ci_high,0.35422
threshold,0.5
verdict,SUPPORTED (material disagreement)
lime_top5_self_agreement,0.876111
shap_top5_seed_agreement,0.926


## SHAP vs LIME on every set and model

In [3]:
pd.read_csv("outputs/d5_shap_vs_lime.csv").round(3)

,model,set,n,tau_median,tau_ci_low,tau_ci_high,tau_mean,top5_feature_agreement_mean,top5_sign_agreement_mean,group_tau_median,primary
0,xgboost,E-rand,1000,0.350,0.344,0.354,0.349,0.536,0.250,1.000,True
1,xgboost,E-top,460,0.347,0.336,0.355,0.340,0.639,0.544,0.333,False
2,xgboost,E-kev,308,0.354,0.340,0.361,0.353,0.527,0.304,1.000,False
3,logreg,E-rand,1000,0.312,0.306,0.318,0.309,0.503,0.252,0.333,False
4,logreg,E-top,460,0.408,0.403,0.415,0.408,0.511,0.426,0.333,False
5,logreg,E-kev,308,0.350,0.342,0.361,0.350,0.505,0.305,0.333,False


## Noise ceilings

In [4]:
pd.read_csv("outputs/d5_noise_ceilings.csv").round(4)

,comparison,n,tau_median,tau_ci_low,tau_ci_high,wilcoxon_shaplime_vs_limelime_p
0,"LIME vs LIME (seeds 1-9 vs 0), M-XGB",200,0.3662,0.3613,0.3705,0.0
1,TreeSHAP vs TreeSHAP (model seeds 1-9 vs 0),1000,0.6604,0.6582,0.6635,NaN
2,SHAP vs LIME on the same 200,200,0.3410,0.3282,0.3529,0.0


**H6 SUPPORTED as registered:** median τ = **0.350 [0.344, 0.354]**, entirely below 0.5. The same holds on E-top (0.347), E-kev (0.354) and for M-LR (0.31–0.41).

**What the noise ceilings add, and why it matters for the write-up:**
- **Over all 69 features, LIME barely agrees with itself:** τ = 0.366 across LIME seeds on the same model and instance. SHAP–LIME (0.341 on the same 200 CVEs) is significantly lower (Wilcoxon p < 1e-10) but only slightly. Most of the full-ranking disagreement is therefore **LIME's sampling noise in the long tail of weak features**, not a systematic conflict. TreeSHAP is more stable across *model* retraining (τ = 0.66).
- **At the top of the explanation, the part an analyst reads, the disagreement is systematic.** LIME's own top-5 is stable across seeds (88% overlap), and TreeSHAP's top-5 is stable across model seeds (93%). Yet SHAP and LIME share only **54%** of their top-5 features, and only **25%** of features are shared *with the same sign*. Two stable methods give different answers.
- Reason: the two methods answer different questions. SHAP attributes the deviation from a background expectation; LIME fits the local slope of P(KEV). A low-EPSS CVE gets a *negative* SHAP value for EPSS (below the background) and a *positive* LIME weight (more EPSS would raise its score). Both are "correct", and neither says which question it answers.
- Group-level agreement (CVSS vs text vs EPSS) is high for M-XGB on E-rand (median τ = 1.0) but not in the top-10 (0.33).

## EXPLORATORY (not pre-registered): SHAP background sensitivity

Added after the D2 case studies showed `cvss_version` dominating every explanation. SHAP on E-rand was recomputed with a **test-era background** (1,000 random test rows, no labels used). The pre-registered results above stand.

In [5]:
pd.read_csv("outputs/dx_background_sensitivity_summary.csv", index_col=0).round(4)

,global_rank_tau_train_vs_test_bg,per_instance_tau_train_vs_test_bg_median,cvss_version_rank_train_bg,cvss_version_rank_test_bg,cvss_version_share_train_bg,cvss_version_share_test_bg,shap_lime_tau_median_test_bg,ci_low,ci_high
xgboost,0.9088,0.7937,1.0,1.0,0.5097,0.1536,0.3248,0.3205,0.3291
logreg,0.9113,0.7110,1.0,5.0,0.3785,0.0364,0.2856,0.2775,0.2899


- `cvss_version` takes **51% of all |SHAP| mass for M-XGB** (38% for M-LR) under the training background, against **15%** (4%) under a test-era background.
- The global ranking otherwise holds: τ ≈ 0.91 between backgrounds. Per-instance rankings move more (median τ 0.79 / 0.71).
- SHAP–LIME agreement stays low under the test-era background (median τ 0.32 / 0.29), so H6 does not hinge on the background choice.
- **Practical rule for deployment:** a SHAP background must come from the population being scored (recent CVEs), not from the training archive. Otherwise about half of every explanation describes the CVE's era.